# A3 01 — Prepare comparable living and bone data

## Purpose
This notebook first refreshes the taxon and weathering rule workbooks from the MSSQL bones_ rule tables, then reads the original census Excel workbooks and defines one reproducible analytical population for all A3 analyses. The primary aerial series includes the sector-level OPC censuses plus Sweetwaters aerial totals for 2000, 2002, and 2003 as historical Eastern observations. Sweetwaters ground/Earthwatch counts from 1996–2003 are prepared separately and are never added to aerial counts. Confirmed blank historical cells are treated as zeros. It does not depend on A2-generated pickle files. Census source workbooks remain read-only, processed data and results are kept separate, and no hypothesis test is performed here.

The shared live–bone taxon set contains wild mammals identified at comparable resolution in both datasets. Cattle, birds, carnivores, unresolved generic rhino, and hybrid zebra are excluded from direct fidelity comparisons. Identified black rhinoceros remains eligible. Unresolved categories remain available for complete bone-only descriptions. Bones collected in low-visibility 2024 are excluded from the primary dataset according to collection date, not estimated death year. The internal fence was removed in March 2007; because bone walks occurred in August and aerial census in September, 2007 is classified as post-removal. Bone death month is unavailable, so any future bone record assigned to 2007 should also be checked in a boundary sensitivity analysis.


## Load and audit the canonical inputs

**Plain-language example.** Before comparing two jars of colored beads, we first record how many beads each jar contains and confirm that the color names mean the same thing. Here, the jars are the aerial and bone Excel files, and the colors are taxa. The import is repeated from those files every time this notebook runs.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))
from a3_common import *
from export_mni_rules import export_rules

processed_dir, output_dir = ensure_a3_directories(ROOT)
RULE_ENV_FILE = ROOT.parent.parent / 'jupyter-env' / '.env'
export_rules(RULE_ENV_FILE, ROOT / 'data/import/excel')

# Change this only when a reviewed normalized weathering-based year field
# has been added to the bone workbook. The selected field must be numeric.
BONE_TIME_COLUMN = 'Year'
aerial_modern, bones_all = load_raw_inputs(ROOT, bone_time_column=BONE_TIME_COLUMN)
aerial_historical, ground_historical = load_historical_living_census(ROOT)
aerial = pd.concat([aerial_modern, aerial_historical], ignore_index=True)
bones_primary = bones_all[bones_all['collection_year'].ne(LOW_VISIBILITY_COLLECTION_YEAR)].copy()
audit = sample_audit(aerial, bones_primary)
audit


C:\LocalData\lintulaa\JupyterWork\BonesOfOlPejeta\projects\bonesofolpejeta\scripts\export_mni_rules.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  found = pd.read_sql_query(
C:\LocalData\lintulaa\JupyterWork\BonesOfOlPejeta\projects\bonesofolpejeta\scripts\export_mni_rules.py:94: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(f"SELECT * FROM {qualified}", connection)


karilint_admin.bones_mnitaxonrule: 61 rows
Columns: ['id', 'source_alias', 'canonical_label', 'default_excluded', 'active', 'notes']
karilint_admin.bones_mniweatheringrule: 12 rows
Columns: ['id', 'source_class', 'canonical_class', 'age_min', 'age_max', 'active', 'reviewed', 'notes', 'age_min_corrected', 'age_max_corrected']


Exported rule workbooks to C:\LocalData\lintulaa\JupyterWork\BonesOfOlPejeta\projects\bonesofolpejeta\data\import\excel


,dataset,sector,year_samples,total_count,median_sample_count,minimum_sample_count,maximum_sample_count,samples_below_10
0,aerial,Eastern,20,43624.0,2078.5,966.0,3973.0,0
1,aerial,Western,17,113891.0,6546.0,4657.0,9853.0,0
2,bones,Eastern,14,466.0,25.0,2.0,87.0,2
3,bones,Western,12,466.0,23.5,2.0,120.0,5


## Define the taxonomic universe

A taxon is eligible for direct fidelity analysis only when it occurs in both source datasets at the same named resolution and is part of the wild-mammal aerial survey population. An absence caused by survey design must not be treated as an ecological zero.

**Plain-language example.** If one survey never looked for lions, zero lions in that survey cannot be compared with lion bones as though both surveys searched equally.


In [2]:
decisions = taxon_decisions(aerial, bones_primary)
shared_taxa = eligible_shared_taxa(aerial, bones_primary)
print(f'Eligible matched taxa: {len(shared_taxa)}')
print(shared_taxa)
decisions


Eligible matched taxa: 15
['Aepyceros melampus', 'Alcelaphus buselaphus', 'Diceros bicornis', 'Equus burchellii', 'Eudorcas thomsonii', 'Giraffa camelopardalis', 'Kobus ellipsiprymnus', 'Loxodonta africana', 'Nanger granti', 'Oryx beisa', 'Phacochoerus africanus', 'Redunca redunca', 'Syncerus caffer', 'Taurotragus oryx', 'Tragelaphus scriptus']


,taxon,in_aerial,in_bones,matched_analysis,decision
0,Acinonyx jubatus,True,False,False,carnivore not consistently represented by aeri...
1,Aepyceros melampus,True,True,True,eligible matched wild-mammal taxon
2,Alcelaphus buselaphus,True,True,True,eligible matched wild-mammal taxon
3,Aves (medium),False,True,False,not observed in both datasets
4,Aves (small),False,True,False,not observed in both datasets
5,Bos taurus indicus,True,True,False,domestic taxon excluded
6,Bovidae (large),False,True,False,unresolved taxonomic category retained only in...
7,Bovidae (medium),False,True,False,unresolved taxonomic category retained only in...
8,Bovidae (small),False,True,False,unresolved taxonomic category retained only in...
9,Canis mesomelas,True,False,False,carnivore not consistently represented by aeri...


## Build and save analysis-ready tables

Bone abundance is expressed as minimum number of individuals (MNI), calculated for each analytical category within each transect. Every transect used the same 1-km route: observers walked from a fixed start to a turning point 1 km away and returned along the same route. Thus, a survey involved 2 km of walking but covered one 1-km route in both directions. Transects were sufficiently separated for their category-level MNI values to be treated as spatially independent and summed within sector × estimated-death-year samples.

Aerial counts are much larger than summed bone MNI, so subsequent fidelity analyses compare within-sample relative composition rather than absolute totals. No distance correction is applied because transect design and route length were standardized. Nevertheless, the number of completed transects can affect summed MNI when sampling effort differs among sector-years. Original abundance values are preserved because small MNI samples provide less precise estimates. Annual matrices are retained for coverage and temporal summaries; period aggregation is performed only in downstream analyses.


In [3]:
aerial_matched = annual_matrix(aerial, shared_taxa)
bones_matched = annual_matrix(bones_primary, shared_taxa)
bones_complete = annual_matrix(bones_primary)
audit = sample_audit(aerial[aerial['Species'].isin(shared_taxa)],
                     bones_primary[bones_primary['Species'].isin(shared_taxa)])

aerial.to_pickle(processed_dir / 'aerial_primary.pkl')
aerial_modern.to_pickle(processed_dir / 'aerial_modern_primary.pkl')
aerial_historical.to_pickle(processed_dir / 'aerial_historical_sweetwaters.pkl')
ground_historical.to_pickle(processed_dir / 'ground_historical_sweetwaters.pkl')
bones_primary.to_pickle(processed_dir / 'bones_primary_excluding_2024.pkl')
bones_all.to_pickle(processed_dir / 'bones_including_2024.pkl')
aerial_matched.to_pickle(processed_dir / 'aerial_matched_annual.pkl')
bones_matched.to_pickle(processed_dir / 'bones_matched_annual.pkl')
bones_complete.to_pickle(processed_dir / 'bones_complete_annual.pkl')
decisions.to_csv(processed_dir / 'taxon_decisions.csv', index=False, encoding='utf-8')
audit.to_csv(processed_dir / 'sampling_audit.csv', index=False, encoding='utf-8')
excluded_taxa = (bones_primary.loc[~bones_primary['Species'].isin(shared_taxa)]
                 .groupby('Species', as_index=False)
                 .agg(records=('Total', 'size'), excluded_MNI=('Total', 'sum'))
                 .sort_values(['excluded_MNI', 'Species'], ascending=[False, True]))
excluded_taxa.to_csv(output_dir / 'A3_01_excluded_bone_taxa.csv', index=False, encoding='utf-8')

summary = pd.DataFrame({
    'item': ['combined aerial rows', 'historical aerial years', 'historical ground years',
             'primary bone rows', '2024-excluded bone rows', 'matched taxa'],
    'value': [len(aerial), aerial_historical['Year'].nunique(), ground_historical['Year'].nunique(),
              len(bones_primary), len(bones_all)-len(bones_primary), len(shared_taxa)]
})
summary


,item,value
0,combined aerial rows,8628
1,historical aerial years,3
2,historical ground years,7
3,primary bone rows,919
4,2024-excluded bone rows,54
5,matched taxa,15


## Preparation interpretation

The revised bone workbook contains 973 transect-level MNI records, of which 919 remain after excluding the 54 records from the low-visibility 2024 collection. Weathering-stage lumping produces 14 primary Eastern and 12 primary Western sector-years before taxonomic matching. The direct comparison contains 15 matched wild-mammal taxa, represented in 14 Eastern and 11 Western sector-years with median summed MNI values of 18 and 16, respectively. Three Sweetwaters aerial totals (2000, 2002, and 2003) extend the primary Eastern aerial series; seven Sweetwaters ground/Earthwatch years (1996–1999 and 2001–2003) are saved as a separate historical living series. Early aerial totals lack block detail, so they support whole-Sweetwaters composition but not block-level analysis. `Reedbuck (Bohor)` is explicitly standardized to the existing database label *Redunca redunca*; other unresolved historical labels are not guessed. Generic `Rhino` and `Zebra Hybrid` are explicitly excluded from direct matching, while identified black rhinoceros remains eligible. Cattle, birds, carnivores, and other unresolved categories remain excluded and are reported with their record and MNI counts in `A3_01_excluded_bone_taxa.csv`. All later notebooks read the processed A3 files without modifying a census workbook.
